In [1]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master("local[4]")
         .appName("TP-RDD")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
print(sc.master, sc.defaultParallelism)

local[4] 4


In [2]:
import os
print(os.getcwd())

/home/jovyan


In [3]:
DATA = "/home/jovyan/work/data"   
os.makedirs(DATA, exist_ok=True)

lignes = """id,date,ville,produit,quantite,prix
1,2025-01-05,Casablanca,Laptop,2,9500
2,2025-01-06,Rabat,Telephone,5,3200
3,2025-01-07,Casablanca,Telephone,3,3200
4,2025-01-09,Fes,Tablette,4,2800
5,2025-01-11,Rabat,Laptop,1,9500
6,2025-01-12,Marrakech,Tablette,6,2800
7,2025-01-15,Casablanca,Souris,20,120
8,2025-01-16,Fes,Laptop,2,9500
9,2025-01-18,Marrakech,Telephone,4,3200
10,2025-01-20,Rabat,Souris,15,120
11,2025-01-21,Agadir,Clavier,10,250
12,ligne_corrompue
13,2025-01-22,Casablanca,Tablette,2,abc
14,2025-01-25,Fes,Souris,30,120
15,2025-01-26,Agadir,Clavier,8,250
16,2025-01-27,Casablanca,Clavier,12,250
17,2025-02-02,Rabat,Telephone,2,3200
18,2025-02-03,Agadir,Laptop,1,9500
19,2025-02-04,Marrakech,Souris,,120
20,2025-02-05,Rabat,Tablette,3,2800
21,2025-02-07,Marrakech,Laptop,2,9500
22,2025-02-09,Fes,Clavier,5,250"""

with open(DATA + "/ventes.csv", "w") as f:
    f.write(lignes + "\n")

print(os.listdir(DATA))

['ventes.csv']


In [4]:
brut = sc.textFile(DATA + "/ventes.csv", 3)
entete = brut.first()
donnees = brut.filter(lambda l: l != entete)

print(entete)
print(brut.count(), donnees.count())
print(brut.getNumPartitions())

id,date,ville,produit,quantite,prix
23 22
3


In [5]:
def parse(ligne):
    try:
        i, d, v, p, q, px = ligne.split(",")
        return [(int(i), d, v, p, int(q), float(px))]
    except (ValueError, IndexError):
        return []

# test local sur quelques lignes, sans Spark
print(parse("1,2025-01-05,Casablanca,Laptop,2,9500"))
print(parse("12,ligne_corrompue"))
print(parse("13,2025-01-22,Casablanca,Tablette,2,abc"))
print(parse("19,2025-02-04,Marrakech,Souris,,120"))

[(1, '2025-01-05', 'Casablanca', 'Laptop', 2, 9500.0)]
[]
[]
[]


In [6]:
ventes = donnees.flatMap(parse).cache()
print(ventes.count())
print(donnees.count() - ventes.count())

19
3


In [7]:
ca = ventes.map(lambda v: (v[2], v[4] * v[5]))
print(ca.take(3))

[('Casablanca', 19000.0), ('Rabat', 16000.0), ('Casablanca', 9600.0)]


In [8]:
ca_ville = ca.reduceByKey(lambda a, b: a + b)
print(ca_ville.collect())

[('Casablanca', 34000.0), ('Fes', 35050.0), ('Rabat', 42100.0), ('Marrakech', 48600.0), ('Agadir', 14000.0)]


In [9]:
g = ca.groupByKey().mapValues(sum)
print(sorted(g.collect()) == sorted(ca_ville.collect()))   # True

print(ca_ville.toDebugString().decode())
print("-----")
print(g.toDebugString().decode())

True
(3) PythonRDD[14] at collect at /tmp/ipykernel_264/4023644942.py:2 []
 |  MapPartitionsRDD[13] at mapPartitions at PythonRDD.scala:168 []
 |  ShuffledRDD[12] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(3) PairwiseRDD[11] at reduceByKey at /tmp/ipykernel_264/4023644942.py:1 []
    |  PythonRDD[10] at reduceByKey at /tmp/ipykernel_264/4023644942.py:1 []
    |  PythonRDD[5] at RDD at PythonRDD.scala:56 []
    |      CachedPartitions: 3; MemorySize: 887.0 B; DiskSize: 0.0 B
    |  /home/jovyan/work/data/ventes.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
    |  /home/jovyan/work/data/ventes.csv HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []
-----
(3) PythonRDD[19] at collect at /tmp/ipykernel_264/1509494460.py:2 []
 |  MapPartitionsRDD[18] at mapPartitions at PythonRDD.scala:168 []
 |  ShuffledRDD[17] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(3) PairwiseRDD[16] at groupByKey at /tmp/ipykernel_264/1509494460.py:1 

In [10]:
nb = ca.countByKey()
print(nb)
print(type(nb))

defaultdict(<class 'int'>, {'Casablanca': 4, 'Rabat': 5, 'Fes': 4, 'Marrakech': 3, 'Agadir': 3})
<class 'collections.defaultdict'>


In [11]:
qmax = (ventes.map(lambda v: (v[3], v[4]))
              .aggregateByKey(0, max, max))
print(qmax.collect())

[('Tablette', 6), ('Clavier', 12), ('Laptop', 2), ('Telephone', 5), ('Souris', 30)]


In [12]:
ca_eur = ca_ville.mapValues(lambda x: x * 0.092)
print(ca_eur.collect())
print(ca_eur.count() == ca_ville.count())
print(ca_eur.getNumPartitions() == ca_ville.getNumPartitions())

[('Casablanca', 3128.0), ('Fes', 3224.6), ('Rabat', 3873.2), ('Marrakech', 4471.2), ('Agadir', 1288.0)]
True
True


In [13]:

t1 = ca_ville.sortBy(lambda kv: kv[1], ascending=False)
print(t1.collect())


t2 = (ca_ville.map(lambda kv: (kv[1], kv[0]))
              .sortByKey(ascending=False)
              .map(lambda kv: (kv[1], kv[0])))
print(t2.collect())

print(t1.collect() == t2.collect())

[('Marrakech', 48600.0), ('Rabat', 42100.0), ('Fes', 35050.0), ('Casablanca', 34000.0), ('Agadir', 14000.0)]
[('Marrakech', 48600.0), ('Rabat', 42100.0), ('Fes', 35050.0), ('Casablanca', 34000.0), ('Agadir', 14000.0)]
True


In [14]:
panier = (ca.mapValues(lambda m: (m, 1))
            .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
            .mapValues(lambda s: s[0] / s[1]))
print(panier.collect())

[('Casablanca', 8500.0), ('Fes', 8762.5), ('Rabat', 8420.0), ('Marrakech', 16200.0), ('Agadir', 4666.666666666667)]


In [15]:
regions = sc.parallelize([
    ("Casablanca", "Casablanca-Settat"), ("Rabat", "Rabat-Sale-Kenitra"),
    ("Fes", "Fes-Meknes"), ("Tanger", "Tanger-Tetouan")], 2)

print(ca_ville.keys().collect())
print(regions.keys().collect())

['Casablanca', 'Fes', 'Rabat', 'Marrakech', 'Agadir']
['Casablanca', 'Rabat', 'Fes', 'Tanger']


In [16]:
j = ca_ville.join(regions)
print(j.collect())

[('Casablanca', (34000.0, 'Casablanca-Settat')), ('Fes', (35050.0, 'Fes-Meknes')), ('Rabat', (42100.0, 'Rabat-Sale-Kenitra'))]


In [17]:
lj = ca_ville.leftOuterJoin(regions)
rj = ca_ville.rightOuterJoin(regions)
print(lj.collect())
print(rj.collect())

[('Casablanca', (34000.0, 'Casablanca-Settat')), ('Fes', (35050.0, 'Fes-Meknes')), ('Rabat', (42100.0, 'Rabat-Sale-Kenitra')), ('Agadir', (14000.0, None)), ('Marrakech', (48600.0, None))]
[('Casablanca', (34000.0, 'Casablanca-Settat')), ('Fes', (35050.0, 'Fes-Meknes')), ('Rabat', (42100.0, 'Rabat-Sale-Kenitra')), ('Tanger', (None, 'Tanger-Tetouan'))]


In [18]:
cg = (ca_ville.cogroup(regions)
              .mapValues(lambda p: (list(p[0]), list(p[1]))))
print(cg.collect())

nb_cles = ca_ville.keys().union(regions.keys()).distinct().count()
print(cg.count(), nb_cles)

[('Casablanca', ([34000.0], ['Casablanca-Settat'])), ('Fes', ([35050.0], ['Fes-Meknes'])), ('Rabat', ([42100.0], ['Rabat-Sale-Kenitra'])), ('Agadir', ([14000.0], [])), ('Tanger', ([], ['Tanger-Tetouan'])), ('Marrakech', ([48600.0], []))]
6 6


In [19]:
rapport = ca_ville.leftOuterJoin(regions).mapValues(lambda p: (p[0], p[1] if p[1] else "Région inconnue"))
print(rapport.collect())

[('Casablanca', (34000.0, 'Casablanca-Settat')), ('Fes', (35050.0, 'Fes-Meknes')), ('Rabat', (42100.0, 'Rabat-Sale-Kenitra')), ('Agadir', (14000.0, 'Région inconnue')), ('Marrakech', (48600.0, 'Région inconnue'))]


In [20]:
pc = ventes.filter(lambda v: v[2] == "Casablanca").map(lambda v: v[3]).distinct()
pr = ventes.filter(lambda v: v[2] == "Rabat").map(lambda v: v[3]).distinct()

print(sorted(pc.collect()))
print(sorted(pr.collect()))
print(sorted(pc.union(pr).distinct().collect()))
print(sorted(pc.intersection(pr).collect()))
print(sorted(pc.subtract(pr).collect()))

['Clavier', 'Laptop', 'Souris', 'Telephone']
['Laptop', 'Souris', 'Tablette', 'Telephone']
['Clavier', 'Laptop', 'Souris', 'Tablette', 'Telephone']
['Laptop', 'Souris', 'Telephone']
['Clavier']


In [21]:
villes = ventes.map(lambda v: v[2]).distinct()
produits = ventes.map(lambda v: v[3]).distinct()

nb_villes, nb_produits = villes.count(), produits.count()
cart = villes.cartesian(produits)
print(nb_villes, nb_produits, cart.count(), nb_villes * nb_produits)

5 5 25 25


In [22]:
a = sc.parallelize(range(1, 6), 2)
b = sc.parallelize(range(11, 16), 2)
print(a.zip(b).collect())

[(1, 11), (2, 12), (3, 13), (4, 14), (5, 15)]


In [23]:
# (i) nombre de partitions différent
b3 = sc.parallelize(range(11, 16), 3)
a.zip(b3).collect()

ValueError: Can only zip with RDD which has the same number of partitions

In [24]:
# (ii) même nombre de partitions, mais 6 éléments au lieu de 5
b6 = sc.parallelize(range(11, 17), 2)
a.zip(b6).collect()

ValueError: Can not deserialize PairRDD with different number of items in batches: (2, 3)

In [25]:
ai = a.zipWithIndex().map(lambda x: (x[1], x[0]))
bi = b6.zipWithIndex().map(lambda x: (x[1], x[0]))
print(ai.join(bi).sortByKey().values().collect())

[(1, 11), (2, 12), (3, 13), (4, 14), (5, 15)]


In [26]:
print(ventes.getNumPartitions())
print(ventes.coalesce(1).getNumPartitions())
print(ventes.repartition(6).getNumPartitions())
print(ventes.coalesce(10).getNumPartitions())

3
1
6
3


In [27]:
print(ventes.coalesce(1).toDebugString().decode())
print("-----")
print(ventes.repartition(2).toDebugString().decode())

(1) CoalescedRDD[179] at coalesce at NativeMethodAccessorImpl.java:0 []
 |  PythonRDD[5] at RDD at PythonRDD.scala:56 []
 |      CachedPartitions: 3; MemorySize: 887.0 B; DiskSize: 0.0 B
 |  /home/jovyan/work/data/ventes.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
 |  /home/jovyan/work/data/ventes.csv HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []
-----
(2) MapPartitionsRDD[184] at coalesce at NativeMethodAccessorImpl.java:0 []
 |  CoalescedRDD[183] at coalesce at NativeMethodAccessorImpl.java:0 []
 |  ShuffledRDD[182] at coalesce at NativeMethodAccessorImpl.java:0 []
 +-(3) MapPartitionsRDD[181] at coalesce at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[180] at RDD at PythonRDD.scala:56 []
    |  PythonRDD[5] at RDD at PythonRDD.scala:56 []
    |      CachedPartitions: 3; MemorySize: 887.0 B; DiskSize: 0.0 B
    |  /home/jovyan/work/data/ventes.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
    |  /home/j

In [28]:
ca_ville.coalesce(1).saveAsTextFile(DATA + "/ca_ville")
print(os.listdir(DATA + "/ca_ville"))
print(sc.textFile(DATA + "/ca_ville").collect())

['part-00000', '._SUCCESS.crc', '_SUCCESS', '.part-00000.crc']
["('Casablanca', 34000.0)", "('Fes', 35050.0)", "('Rabat', 42100.0)", "('Marrakech', 48600.0)", "('Agadir', 14000.0)"]


In [30]:
import shutil
shutil.rmtree(DATA + "/ca_ville")

FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/work/data/ca_ville'

In [31]:
ventes.map(lambda v: (v[3], v[4])) \
      .reduceByKey(lambda a, b: a + b) \
      .max(key=lambda kv: kv[1])

('Souris', 65)

In [32]:
(ventes.map(lambda v: (v[2], (v[4] * v[5], 1)))
       .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
       .mapValues(lambda s: s[0] / s[1])
       .max(key=lambda kv: kv[1]))

('Marrakech', 16200.0)

In [33]:
(ventes.map(lambda v: (v[3], v[2]))
       .distinct()
       .groupByKey()
       .mapValues(sorted)
       .collect())

[('Tablette', ['Fes', 'Marrakech', 'Rabat']),
 ('Clavier', ['Agadir', 'Casablanca', 'Fes']),
 ('Laptop', ['Agadir', 'Casablanca', 'Fes', 'Marrakech', 'Rabat']),
 ('Telephone', ['Casablanca', 'Marrakech', 'Rabat']),
 ('Souris', ['Casablanca', 'Fes', 'Rabat'])]

In [34]:
moy = ventes.map(lambda v: v[4] * v[5]).mean()
print(moy)
ventes.filter(lambda v: v[4] * v[5] > moy).map(lambda v: v[0]).collect()

9144.736842105263


[1, 2, 3, 4, 5, 6, 8, 9, 18, 21]

In [35]:
ca_ville = ca.reduceByKey(lambda a, b: a + b)
print(ca_ville.collect())


[('Casablanca', 34000.0), ('Fes', 35050.0), ('Rabat', 42100.0), ('Marrakech', 48600.0), ('Agadir', 14000.0)]
